In [1]:
# conda env 'vis2_clone'
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent.parent.parent

import sys
import os
sys.path.append(str(PROJECT_ROOT))

from src.utils.util import load_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import glob
import at3d
import subprocess
import paramiko

from collections import defaultdict

In [2]:
def get_slurm_ID(filepath):
    result = subprocess.run(['git', 'log', '--grep={}'.format(filepath)], capture_output=True, text=True)
    assert len(result.stdout.split('\n')) > 5, "Slurm job not found: {}".format(filepath)
    return result.stdout.split('\n')[4].split(' ')[-1]

def get_node(SID):
    result = subprocess.run(['sacct', '-j', SID, '--format=JobID,JobName,NodeList'], capture_output=True, text=True)
    assert len(result.stdout.split('\n')) > 1, "Job not found"
    return result.stdout.split('\n')[2].split()[-1]

def get_cpu_model(node):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) 
    ssh.connect(node)
    stdin, stdout, stderr = ssh.exec_command('lscpu | grep "Model name:"')
    out_str = stdout.read().decode('utf-8')
    ssh.close()
    assert len(out_str) > 0, "CPU not available"
    return out_str.split(':')[-1].strip()

def parse_at3d_output(dir):
    loss_file = dir / 'loss.csv'
    log_file = dir / 'at3d.log'
    info = {}
    config = load_config(dir / 'config.yml')
    try:
        with open(log_file, 'r') as file:
            lines = list(file)
            info['termination'] = lines[-1].strip()
            info['F'] = float(lines[-3].split()[2])
            info['N'] = int(lines[-4].split()[0])
            info['Tit'] = int(lines[-4].split()[1])
            info['Tnf'] = int(lines[-4].split()[2])
            info['Tnint'] =  int(lines[-4].split()[3])
            info['Skip'] = int(lines[-4].split()[4])
            info['Nact'] = int(lines[-4].split()[5])
            info['Projg'] = float(lines[-4].split()[6].replace('D','E'))
            info['max_cell_splitting'] = False
            info['slurm_ID'] = get_slurm_ID(config['out_path'])
            info['node'] = get_node(info['slurm_ID'])
            info['cpu'] = get_cpu_model(info['node'])
            for l in lines:
                if('Further cell splitting' in l):
                    info['max_cell_splitting'] = True
                if('Execution time' in l):
                    info['run time'] = float(l.split()[2])
    except FileNotFoundError:
        print('Error: File not found at {}'.format(log_file))
    except Exception as e:
        print(f"Error while parsing log file: {e}")

    try:
        with open(loss_file, 'r') as file:
            lines = list(file)
            info['total_obj_fn_time'] = float(lines[-1].split()[2])
    except FileNotFoundError:
        print('Error: File not found at {}'.format(loss_file))
    except Exception as e:
        print(f"Error while parsing log file: {e}")
    
    return info

In [3]:
def load_retreival(dir):
    iterations = dir / 'iterations'
    assert iterations.is_dir()
    files = sorted(iterations.glob('iter*.nc'))
    sensors, solvers, rte_grid = at3d.util.load_forward_model(files[-1])
    data = {
        'files': files,
        'sensors': sensors,
        'solvers': solvers,
        'rte_grid': rte_grid,
        'log': parse_at3d_output(dir),
    }
    return data

In [4]:
def traverse(dir, data, parent_key=()):
    name = dir.name
    parent_key += (name,)
    iterations = dir / 'iterations'
    if(iterations.is_dir()):
        data[parent_key] = load_retreival(dir)
    else:
        for n in dir.iterdir():
            traverse(dir / n, data, parent_key)


In [5]:
V = 'V11'
VERSION = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / V
DEFAULT = VERSION / 'default'

config = load_config(DEFAULT / 'config.yml')

In [6]:
data = defaultdict(defaultdict)
traverse(VERSION, data)

/data/keeling/a/johntl3/miniconda3/envs/vis2_clone/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2_clone/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '155'. Max cell optical depth: '4.251702785491943'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "
/data/keeling/a/johntl3/miniconda3/envs/vis2_clone/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/env

AttributeError: module 'numpy' has no attribute 'int'.
`np.int` was a deprecated alias for the builtin `int`. To avoid this error in existing code, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [ ]:
def print_nested_dictionary_keys(nested_dict, indent=0):
    """
    Recursively prints keys from a nested dictionary with indentation.

    Args:
        nested_dict (dict): The nested dictionary to print keys from.
        indent (int): The current indentation level.
    """
    for key, value in nested_dict.items():
        print('  ' * indent + str(key)) # Print key with indentation
        if isinstance(value, dict):
            print_nested_dictionary_keys(value, indent + 1) # Recursive call for nested dictionaries

print_nested_dictionary_keys(data)

### Error Metrics

$\bm{\beta_p}$ is the perturbed extinction coefficient vector. $\bm{\beta_r}$ is the reference extinction coefficient vector.

Relative Bias (Jesse's relative bias):

$\text{Relative Bias} = 100 \times \frac{||\bm{\beta_p}||_1 - ||\bm{\beta_r}||_1}{||\bm{\beta_r}||_1}$

Relative Difference (Jesse's relative RMSE):

$\text{Relative Difference} = 100 \times \frac{||\bm{\beta_p} - \bm{\beta_r}||_2}{||\bm{\beta_r}||_2}$

$||\bm{x}||_1 = \sum_i |x_i|$ and $||\bm{x}||_2 = \left(\sum_i x_i^2\right)^{\frac{1}{2}}$

Mean Absolute Error:

$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |\beta_{p,i} - \beta_{r, i}|$

Root Mean Square Error:

$\text{RMSE} = \left(\frac{1}{n} \sum_{i=1}^n \left(\beta_{p,i} - \beta_{r,i}\right)^2\right)^\frac{1}{2} = \left(\frac{1}{n} ||\bm{\beta_p} - \bm{\beta_r}||_2^2\right)^{\frac{1}{2}}$

Correlation Coefficient (Pearson's r):

$r = \frac{\sum_i (\beta_{p,i} - \bar{\bm{\beta_p}})(\beta_{r,i} - \bar{\bm{\beta_r}})}{\sqrt{\sum_i (\beta_{p,i} - \bar{\bm{\beta_p}})^2} \sqrt{\sum_i (\beta_{r,i} - \bar{\bm{\beta_r}})^2}}$

Normalized Mean Bias (NMB):

$\text{NMB} = \frac{\sum_i (\beta_{p,i} - \beta_{r,i})}{\sum_i \beta_{r,i}}$


In [ ]:
def calc_error_metrics(reference, data):
    if('solvers' in list(data.keys())):
        ext_tmp = np.ravel(data['solvers'][0.67].medium['cloud'].extinction.data)
        ext_ref = np.ravel(reference['solvers'][0.67].medium['cloud'].extinction.data)
        data['relative difference'] = 100 * (np.sum((ext_tmp - ext_ref)**2))**0.5 / (np.sum(ext_ref)**2)**0.5
        data['relative bias'] = 100 * (np.sum(np.abs(ext_tmp)) - np.sum(np.abs(ext_ref))) / (np.sum(np.abs(ext_ref)))
        data['MAE'] = np.sum(np.abs(ext_tmp - ext_ref)) / len(ext_tmp)
        data['RMSE'] = (np.sum((ext_tmp - ext_ref)**2) / len(ext_tmp))**0.5
        data['r'] = (np.sum((ext_tmp - np.mean(ext_tmp)) * (ext_ref - np.mean(ext_ref)))) / ((np.sum((ext_tmp - np.mean(ext_tmp))**2)**0.5) * (np.sum((ext_ref - np.mean(ext_ref))**2)**0.5))
        data['NMB'] = np.sum(ext_tmp - ext_ref) / np.sum(ext_ref)

        tmp = data['solvers'][0.67].medium['cloud'].copy(deep=True)
        mask = (tmp.extinction.data > 0).astype(np.int64)
        tmp['cvm'] = (['x', 'y', 'z'], mask)
        cv = tmp.cvm.integrate(['x','y','z']).data
        tmp_ref = reference['solvers'][0.67].medium['cloud'].copy(deep=True)
        mask = (tmp_ref.extinction.data > 0).astype(np.int64)
        tmp_ref['cvm'] = (['x', 'y', 'z'], mask)
        cv_ref = tmp_ref.cvm.integrate(['x','y','z']).data
        data['cloud volume'] = cv
        data['delta volume'] = cv - cv_ref

        tmp['tau'] = tmp.extinction.integrate(['z'])
        tmp['cloud_mask'] = (tmp.tau > 0).astype(np.int64)
        tmp['cf'] = tmp.cloud_mask.sum() / (tmp.sizes['x'] * tmp.sizes['y'])
        data['cloud fraction'] = tmp.cf.data
        data['tau bar'] = tmp.tau.where(tmp.cloud_mask > 0).mean().data
    else:
        for n in data:
            calc_error_metrics(reference, data[n])

In [ ]:
reference = data[(V, 'default')]
calc_error_metrics(reference, data)

In [ ]:
cvm = xr.load_dataset(config['cloud_volume_mask_file'])
vol_cvm = cvm.cloud_volume_mask.integrate(['x','y','z']).data
print('Default CVM Volume: {:.3f} km**3'.format(vol_cvm))

In [ ]:
print('{:<53} | {:<8} | {:<8} | {:<6} | {:<7} | {:<9} '.format('Path', 'Bias (%)', 'Diff (%)', 'CF', 'tau bar', 'V (km**3)'))
print('-'*100)
for k, v in data.items():
    print_line = True
    for i in k:
        if('obsolete' in i):
            print_line = False
    if(print_line):
        print('{:<53} | {:<8.2f} | {:<8.2f} | {:<6.3f} | {:<7.3f} | {:<9.2f}'.format(str(k), v['relative bias'], v['relative difference'], v['cloud fraction'], v['tau bar'], v['cloud volume']))

In [ ]:
print('{:<53} | {:<11} | {:<41} | {:<5} | {:<4} | {:<6} | {:<6} | {:<7} | {:<6} | {:<14} | {:<20} '.format(
    'Path',
    # 'SID',
    'node',
    'cpu',
    'N',
    'Tit',
    't (s)',
    's/iter',
    '% SHDOM',
    'F',
    'Max Cell Split',
    'Termination'
))
print('-'*220)
for k, v in data.items():
    if('Tit' in v['log']):
        print('{:<53} | {:<11} | {:<41} | {:<5d} | {:<4d} | {:<6.0f} | {:<6.1f} | {:<7.2f} | {:<6.2f} | {:<14} | {:<20} '.format(
            str(k),
            # v['log']['slurm_ID'],
            v['log']['node'],
            v['log']['cpu'],
            v['log']['N'],
            v['log']['Tit'],
            v['log']['run time'],
            v['log']['run time']/v['log']['Tit'],
            100 - 100*v['log']['total_obj_fn_time']/v['log']['run time'],
            v['log']['F'],
            v['log']['max_cell_splitting'],
            v['log']['termination']
        ))